# 06 — Document Loaders
### LangChain Retrieval · Part 1 of 4

Until now, every piece of text the model saw was something **you typed into the notebook**. Real
applications don't work that way: the knowledge lives in PDFs, spreadsheets, web pages, and folders
of files. A **document loader** is the piece that goes and gets it, and hands it to LangChain in one
standard shape — so everything downstream (splitting, embedding, searching) doesn't care whether the
text came from a PDF or a website.

> 📌 **Good to know (2026)**
> - ⚠️ **`langchain-community` was sunset in May 2026.** The package these loaders come from is now
>   frozen: no new features, no maintenance, repo archived. LangChain's direction is *one package per
>   provider* (`langchain-openai`, `langchain-chroma`, ...). Every loader below still runs — this
>   notebook was tested against `langchain-community 0.4.2` — and they're perfectly good for learning
>   the concepts, which don't change. For a **production** system, check the integrations page for a
>   maintained standalone package before depending on one, and keep loader code behind a small
>   function of your own (the capstone's `ingest_knowledge_base` does exactly that), so swapping a
>   loader later touches one place instead of your whole app.
> - There isn't one "PDF loader". The docs list around ten (PyPDF, PyMuPDF, PDFPlumber, Unstructured,
>   MathPix, ...) because PDFs are messy. `PyPDFLoader` is the right *starting* point for text PDFs;
>   scanned or layout-heavy PDFs need something else (table in Section 4).
> - `PyPDFLoader(..., extract_images=True)` pulls in an OCR dependency and has been reported to crash
>   on some PDFs. Leave it off unless you specifically need text from images.
> - `WebBaseLoader` wants a `USER_AGENT` environment variable, or it prints a warning on every run.

## The story of this notebook

The scripts you wrote earlier in this folder all follow the same pattern: load a file, paste
**everything** into the prompt, ask a question. That works — and this notebook shows exactly why it
stops working. The last section measures the three walls you hit as your data grows, which is the
motivation for `07_Text_Splitters/` through `09_Retriever/`.

## Setup

In [1]:
%pip install -q langchain langchain-core langchain-groq python-dotenv langchain-community pypdf beautifulsoup4
!pip install pypdf 
!pip install beautifulsoup4

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from datetime import datetime, timezone
from pathlib import Path
from dotenv import load_dotenv
load_dotenv()

os.environ.setdefault("USER_AGENT", "genai-learning-notebook")   # silences WebBaseLoader's warning

from langchain_core.documents import Document
from langchain_community.document_loaders import (
    TextLoader, CSVLoader, PyPDFLoader, WebBaseLoader, DirectoryLoader,
)
print("Environment ready.")


C:\Users\Deepak Kumar\AppData\Local\Temp\ipykernel_30452\2591799811.py:10: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import (
d:\Gen Ai\myvenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Environment ready.


## Make the notebook self-contained: create our own sample files

Your earlier scripts pointed at hardcoded paths like `D:\Gen Ai\Documents_Loader\documents\...`.
That works on one machine and breaks on every other one (including a teammate's, or GitHub Actions).
The habit to build: **paths relative to the project, built with `pathlib`**. This cell creates a small
`sample_docs/` folder next to the notebook — a customer-support knowledge base, the same world we've
used since `01_Models`.

In [3]:
DOCS_DIR = Path("sample_docs")
DOCS_DIR.mkdir(exist_ok=True)

(DOCS_DIR / "returns_policy.txt").write_text("""Returns Policy

Customers may return most items within 30 days of delivery. Items must be unused and in their original packaging.

Refunds are issued to the original payment method within 5-7 business days after we receive the returned item. Shipping fees are non-refundable unless the item arrived damaged or defective.

Digital products and gift cards cannot be returned once delivered.
""", encoding="utf-8")

(DOCS_DIR / "shipping_policy.txt").write_text("""Shipping Policy

Standard shipping takes 3-5 business days. Express shipping takes 1-2 business days and is available for orders placed before 2 PM.

Orders over $50 qualify for free standard shipping. International shipping is available to select countries and typically takes 7-14 business days.
""", encoding="utf-8")

(DOCS_DIR / "support_faq.csv").write_text("""question,answer,category
How do I request a refund?,"Open Orders, choose the item, and select Return. Your refund is issued within 5-7 business days after we receive the item.",Billing
Can I change my delivery address?,You can change the address before the order ships from the Orders page.,Shipping
Why was I charged twice?,A temporary authorization hold can look like a duplicate charge. It drops off automatically within 3 days.,Billing
How do I reset my password?,Choose Forgot password on the sign-in page and follow the emailed link.,Account
Do you ship internationally?,"Yes, to select countries. International orders take 7-14 business days.",Shipping
How do I cancel my subscription?,Go to Settings > Subscription > Cancel. Access continues until the end of the billing period.,Billing
What payment methods do you accept?,"We accept major credit cards, debit cards, and UPI.",Billing
How do I contact support?,Use the chat widget or email support@example.com. Replies arrive within 24 hours.,Account
""", encoding="utf-8")

print(sorted(p.name for p in DOCS_DIR.iterdir()))


['returns_policy.txt', 'shipping_policy.txt', 'support_faq.csv']


---
## 1. The `Document` — the one shape everything shares

Every loader, whatever it reads, returns a **list of `Document` objects**. A `Document` is just two
things:

| Field | What it holds | Why it matters |
|---|---|---|
| `page_content` | The text | This is what gets embedded and shown to the model |
| `metadata` | A dict — source file, page number, row, URL, anything you add | This is how you **cite sources** and **filter** later ("only search HR documents") |

You can build one by hand, which is exactly what you do when your data comes from a **database or an
API** instead of a file — there's no loader for your company's internal system, so you make the
Documents yourself.

In [4]:
doc = Document(
    page_content="Refunds are issued within 5-7 business days.",
    metadata={"source": "manual", "department": "support"},
)
print(doc.page_content)
print(doc.metadata)


Refunds are issued within 5-7 business days.
{'source': 'manual', 'department': 'support'}


In [5]:
# Real-world: turning rows from a database / API response into Documents
ticket_rows = [
    {"id": 101, "subject": "Card charged twice", "resolution": "Authorization hold; released in 3 days.", "team": "billing"},
    {"id": 102, "subject": "Can't sign in",      "resolution": "Password reset link sent and confirmed.",  "team": "account"},
]

ticket_docs = [
    Document(
        page_content=f"Subject: {r['subject']}\nResolution: {r['resolution']}",
        metadata={"source": f"ticket-{r['id']}", "team": r["team"]},
    )
    for r in ticket_rows
]
print(len(ticket_docs), "documents built from rows")
print(ticket_docs[0].metadata)


2 documents built from rows
{'source': 'ticket-101', 'team': 'billing'}


---
## 2. `TextLoader` — one file, one `Document`

The simplest loader. Two things to know: **always pass `encoding="utf-8"`** (on Windows the default
encoding is not UTF-8, and a file with a curly quote or a Hindi word will crash), and it returns
**one Document per file**, however long the file is.

In [6]:
loader = TextLoader(DOCS_DIR / "returns_policy.txt", encoding="utf-8")
docs = loader.load()

print("Number of Documents:", len(docs))      # 1 — one file = one Document
print("Metadata:", docs[0].metadata)
print("Characters:", len(docs[0].page_content))
print()
print(docs[0].page_content)


Number of Documents: 1
Metadata: {'source': 'sample_docs\\returns_policy.txt'}
Characters: 389

Returns Policy

Customers may return most items within 30 days of delivery. Items must be unused and in their original packaging.

Refunds are issued to the original payment method within 5-7 business days after we receive the returned item. Shipping fees are non-refundable unless the item arrived damaged or defective.

Digital products and gift cards cannot be returned once delivered.



---
## 3. `CSVLoader` — one row, one `Document`

The opposite granularity from `TextLoader`: **every row becomes its own Document**, with the column
names baked into the text (`question: ...`, `answer: ...`). That's usually what you want for an FAQ —
each Q&A pair is a self-contained unit that can be found and cited on its own.

Two useful options: `metadata_columns` copies chosen columns into `metadata` (great for filtering
later), and `source_column` makes each row's `source` something more useful than the file path.

In [7]:
loader = CSVLoader(
    file_path=DOCS_DIR / "support_faq.csv",
    encoding="utf-8",
    metadata_columns=["category"],     # copy this column into metadata for later filtering
)
faq_docs = loader.load()

print("Number of Documents:", len(faq_docs))    # 8 — one per row
print("Metadata:", faq_docs[0].metadata)         # source, row number, and our category column
print()
print(faq_docs[0].page_content)


Number of Documents: 8
Metadata: {'source': 'sample_docs\\support_faq.csv', 'row': 0, 'category': 'Billing'}

question: How do I request a refund?
answer: Open Orders, choose the item, and select Return. Your refund is issued within 5-7 business days after we receive the item.


---
## 4. `PyPDFLoader` — one page, one `Document`

PDFs split **by page**, and each Document's metadata records the page number — which is what lets a
chatbot later say *"see page 14"*. One trap: `metadata["page"]` is a **zero-based index** (the first
page is `0`), while `metadata["page_label"]` is the number printed on the page. Cite the label, or
your "page 14" will point at page 15.

> ⚠️ **A mistake worth avoiding:** `docs[0].page_content` is **only the first page**. Passing just
> that into a prompt for a 40-page PDF silently throws away 39 pages, and the model will happily
> answer from page 1 alone (or say "I couldn't find it") without any error. Always think in terms of
> the *whole list*.

This cell looks for any PDF you drop into `sample_docs/` (the handbook, a resume, a paper — whatever
you have), and skips politely if there isn't one.

In [9]:
pdf_files = sorted(DOCS_DIR.glob("*.pdf"))

if pdf_files:
    pdf_docs = PyPDFLoader(str(pdf_files[0])).load()
    print(f"{pdf_files[0].name}: {len(pdf_docs)} pages -> {len(pdf_docs)} Documents")
    meta = pdf_docs[0].metadata
    # 'page' is ZERO-based (an index); 'page_label' is the number a human sees printed on the page
    print("page:", meta["page"], "| page_label:", meta["page_label"], "| total_pages:", meta["total_pages"])
    print("Page 1 preview :", pdf_docs[0].page_content[:200].replace("\n", " "), "...")
else:
    pdf_docs = []
    print("No PDF found in sample_docs/. Copy any text-based PDF in there and re-run this cell.")


AI Agents.pdf: 117 pages -> 117 Documents
page: 0 | page_label: 1 | total_pages: 117
Page 1 preview : FREE AI AGENTS 2025 EDITION THE ILLUSTRATED GUIDEBOOK  Avi Chawla & Akshay Pachaar DailyDoseofDS.com Daily Dose of Data Science ...


### Which PDF loader should you use?

| Loader | Reach for it when |
|---|---|
| `PyPDFLoader` | Normal, text-based PDFs. The default starting point |
| `PyMuPDFLoader` | You want speed and better handling of tricky files |
| PyMuPDF4LLM integration | You want **Markdown output** that keeps headings and structure (separate integration — check the docs page for its current install and class name) |
| `PDFPlumberLoader` | The PDF is full of **tables** |
| `UnstructuredPDFLoader` | Complex layouts, or **scanned PDFs** (needs OCR) |

**How to tell you have a scanned PDF:** the loader runs without error but `page_content` is empty or
gibberish. A scanned PDF is really a stack of pictures of pages — there is no text to extract until
OCR reads the images. (Given the `langchain-community` sunset above, check the current integrations
page for install instructions before picking one.)

---
## 5. `WebBaseLoader` — a web page, one `Document`

Loads a URL and extracts its text using BeautifulSoup. The catch: a raw web page is mostly **junk** —
navigation menus, cookie banners, footers, ads — and all of it lands in `page_content` and later gets
embedded and paid for. The fix is a `SoupStrainer`, which tells the parser to keep **only** the tags
you name.

The cell below needs internet access, so it's wrapped to fail gracefully.

In [10]:
import bs4

URL = "https://reference.langchain.com/python/langchain-community/document_loaders"

try:
    # Only keep headings, paragraphs and list items — drops menus, sidebars and footers
    only_content = bs4.SoupStrainer(["h1", "h2", "h3", "p", "li"])

    web_docs = WebBaseLoader(
        web_paths=[URL],
        bs_kwargs={"parse_only": only_content},
    ).load()

    print("Characters kept:", len(web_docs[0].page_content))
    print("Metadata:", web_docs[0].metadata)
    print(web_docs[0].page_content[:300])
except Exception as e:
    web_docs = []
    print("Could not load the page (offline or blocked?):", type(e).__name__)


Characters kept: 42151
Metadata: {'source': 'https://reference.langchain.com/python/langchain-community/document_loaders'}
Ask AIGitHubMain DocsOverviewLangChain AssistantAsk a question to get startedEnter to send•Shift+Enter new lineMenudocument_loadersDocument Loaders  are classes to load Documents.Document Loaders are usually used to load a lot of Documents in a single run.Class hierarchy:.. code-block::Main helpers:


In [11]:
# The SoupStrainer idea, demonstrated offline so you can see exactly what it does
html = """
<nav>Home | Docs | Pricing | Login</nav>
<h1>Refund Policy</h1>
<p>Refunds take 5-7 business days.</p>
<footer>Copyright 2026 - Cookie settings - Privacy</footer>
"""

everything = bs4.BeautifulSoup(html, "html.parser").get_text(" ", strip=True)
strained   = bs4.BeautifulSoup(html, "html.parser", parse_only=bs4.SoupStrainer(["h1", "p"])).get_text(" ", strip=True)

print("Without strainer:", everything)
print("With strainer   :", strained)


Without strainer: Home | Docs | Pricing | Login Refund Policy Refunds take 5-7 business days. Copyright 2026 - Cookie settings - Privacy
With strainer   : Refund Policy Refunds take 5-7 business days.


---
## 6. `DirectoryLoader` — a whole folder at once

Real knowledge bases are folders, not single files. `DirectoryLoader` walks a directory, matches
files with a glob pattern, and applies whichever loader you name to each one. It applies **one
loader type per call**, so a mixed folder needs one `DirectoryLoader` per file type.

In [12]:
txt_docs = DirectoryLoader(
    str(DOCS_DIR),
    glob="**/*.txt",                       # ** means "including sub-folders"
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},   # passed to each TextLoader
).load()

print(len(txt_docs), "text files loaded:")
for d in txt_docs:
    print("  -", Path(d.metadata["source"]).name)


2 text files loaded:
  - returns_policy.txt
  - shipping_policy.txt


---
## 7. Lazy loading — for when the data doesn't fit in memory

`.load()` reads **everything into a list at once**. Fine for a folder of policies; a problem for
50,000 PDFs. `.lazy_load()` returns a **generator** instead — it hands you one Document at a time and
lets you process and discard each before the next is read, so memory stays flat no matter how big the
corpus is.

In [13]:
lazy = DirectoryLoader(
    str(DOCS_DIR), glob="**/*.txt", loader_cls=TextLoader, loader_kwargs={"encoding": "utf-8"},
).lazy_load()

print(type(lazy).__name__)     # generator — nothing has been read yet

total_chars = 0
for d in lazy:                 # files are read one at a time, here
    total_chars += len(d.page_content)     # process it, then let it go
print("Total characters seen:", total_chars)


generator
Total characters seen: 687


---
## 8. Enrich the metadata *now*, while you still know where everything came from

Loaders give you `source` and maybe `page` or `row`. That's rarely enough. Once documents are
chopped into chunks and embedded (next three notebooks), you can no longer ask "which department
owns this?" — unless you wrote it down **at load time**. Metadata added here follows every chunk all
the way through, and later lets you filter a search (*"only look at billing documents"*) and show
users where an answer came from.

In [14]:
for d in faq_docs:
    d.metadata["department"] = "support"
    d.metadata["doc_type"] = "faq"
    d.metadata["ingested_at"] = datetime.now(timezone.utc).isoformat(timespec="seconds")

print(faq_docs[0].metadata)


{'source': 'sample_docs\\support_faq.csv', 'row': 0, 'category': 'Billing', 'department': 'support', 'doc_type': 'faq', 'ingested_at': '2026-10-03T05:05:58+00:00'}


---
## Real-world capstone: a knowledge-base ingestion pipeline — and the three walls

Everything above, packaged as one function you'd actually keep: point it at a folder, get back a clean
list of enriched Documents plus a report of what was loaded.

In [15]:
def ingest_knowledge_base(folder: Path, department: str) -> list[Document]:
    """Load every .txt, .csv and .pdf under `folder` and stamp consistent metadata on each Document."""
    loaders = [
        DirectoryLoader(str(folder), glob="**/*.txt", loader_cls=TextLoader, loader_kwargs={"encoding": "utf-8"}),
        DirectoryLoader(str(folder), glob="**/*.csv", loader_cls=CSVLoader,  loader_kwargs={"encoding": "utf-8"}),
        DirectoryLoader(str(folder), glob="**/*.pdf", loader_cls=PyPDFLoader),
    ]

    stamp = datetime.now(timezone.utc).isoformat(timespec="seconds")
    documents: list[Document] = []
    for loader in loaders:
        documents.extend(loader.load())

    for d in documents:
        d.metadata["department"] = department
        d.metadata["doc_type"] = Path(d.metadata["source"]).suffix.lstrip(".")
        d.metadata["ingested_at"] = stamp
    return documents


def report(documents: list[Document]) -> None:
    by_type: dict[str, dict] = {}
    for d in documents:
        s = by_type.setdefault(d.metadata["doc_type"], {"docs": 0, "chars": 0})
        s["docs"] += 1
        s["chars"] += len(d.page_content)

    print(f"{'type':<6}{'documents':>10}{'characters':>12}{'~tokens':>10}")
    for t, s in by_type.items():
        print(f"{t:<6}{s['docs']:>10}{s['chars']:>12}{s['chars'] // 4:>10}")
    total = sum(s["chars"] for s in by_type.values())
    print(f"{'TOTAL':<6}{len(documents):>10}{total:>12}{total // 4:>10}   (1 token ≈ 4 characters)")


knowledge_base = ingest_knowledge_base(DOCS_DIR, department="support")
report(knowledge_base)


type   documents  characters   ~tokens
txt            2         687       171
csv            8        1194       298
pdf          117       46129     11532
TOTAL        127       48010     12002   (1 token ≈ 4 characters)


### The three walls of "just paste everything into the prompt"

This is the approach your earlier loader scripts used, and on 3 tiny files it works fine. Here is
exactly where it breaks.

**Wall 1 — keyword matching is brittle.** Your CSV script filtered rows with
`"favorite season" in doc.page_content`. Watch what happens when a real customer phrases the same
question in their own words.

In [16]:
STOPWORDS = {"the", "and", "how", "do", "i", "my", "to", "a", "is", "can", "you", "get", "your"}

def keyword_search(query: str, documents: list[Document]) -> list[Document]:
    words = [w for w in query.lower().split() if w not in STOPWORDS and len(w) > 2]
    return [d for d in documents if any(w in d.page_content.lower() for w in words)]

for query in ["refund", "get my money back"]:
    hits = keyword_search(query, knowledge_base)
    print(f"{query!r:24} -> {len(hits)} matching documents")


'refund'                 -> 2 matching documents
'get my money back'      -> 8 matching documents


Same intent, **zero results** — because the customer said *"money back"* and the documents say
*"refund"*. Keyword matching only finds text that shares your exact words. The fix is searching by
**meaning**, using the embeddings you met in `01_Models/02_embedding_models.ipynb` — that's
`08_Vector_Store/` and `09_Retriever/`.

**Wall 2 — the model sees everything, so it can handle the wording.** Ask the same question and hand
the model the whole knowledge base, and it succeeds — because a language model understands "money
back" ≈ "refund". Note the small habit worth keeping: each chunk of context is labelled with its
`source`, so the answer can cite where it came from.

In [25]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_groq import ChatGroq
from langchain_openai import ChatOpenAI



def format_docs(documents: list[Document]) -> str:
    return "\n\n".join(
        f"[source: {Path(d.metadata['source']).name}]\n{d.page_content}" for d in documents
    )

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer using ONLY the context below. Cite the source file name in brackets. "
               "If the answer is not in the context, say you could not find it.\n\nContext:\n{context}"),
    ("human", "{question}"),
])

qa_chain = qa_prompt | ChatOpenAI(model="gpt-4o-mini", temperature=0) | StrOutputParser()

print(qa_chain.invoke({
    "context": format_docs(knowledge_base),
    "question": "How do I get my money back?",
}))


To request a refund, open Orders, choose the item, and select Return. Your refund is issued within 5-7 business days after we receive the item. [support_faq.csv]


**Wall 3 — it doesn't scale.** Sending the whole knowledge base works here because it's roughly
500 tokens. Scale it to something real and the same code falls over, quietly and expensively.

In [26]:
current_tokens = sum(len(d.page_content) for d in knowledge_base) // 4

# A 200-page employee handbook: ~500 words/page * ~1.3 tokens/word
handbook_tokens = 200 * 500 * 13 // 10

print(f"Our whole knowledge base today : ~{current_tokens:,} tokens per question")
print(f"One 200-page handbook          : ~{handbook_tokens:,} tokens per question")
print(f"A 50-document company wiki     : ~{50 * handbook_tokens:,} tokens per question")
print()
print("Every single question re-sends ALL of it. That means:")
print("  1. Cost   - you pay for every token, on every question")
print("  2. Limit  - past the model's context window it simply won't fit")
print("  3. Focus  - burying one relevant paragraph in 130,000 tokens makes answers worse, not better")


Our whole knowledge base today : ~12,002 tokens per question
One 200-page handbook          : ~130,000 tokens per question
A 50-document company wiki     : ~6,500,000 tokens per question

Every single question re-sends ALL of it. That means:
  1. Cost   - you pay for every token, on every question
  2. Limit  - past the model's context window it simply won't fit
  3. Focus  - burying one relevant paragraph in 130,000 tokens makes answers worse, not better


So the loading step is done, but the *approach* has to change: instead of sending everything, send
only the few passages relevant to the question. Getting there takes three steps, one notebook each:

1. **`07_Text_Splitters`** — cut Documents into small, meaningful chunks
2. **`08_Vector_Store`** — embed the chunks and store them so they can be searched by meaning
3. **`09_Retriever`** — fetch the best few chunks for each question

### Try it yourself
1. Drop a PDF into `sample_docs/`, re-run `ingest_knowledge_base`, and check the report — how many
   Documents did it become, and does the page count match?
2. Add a `.md` file to the folder and add a fourth `DirectoryLoader` for it (`glob="**/*.md"` with
   `TextLoader`).
3. Add `source_column="category"` to the `CSVLoader` and look at how `metadata["source"]` changes —
   when would that be useful?
4. Write your own keyword query that *does* match a document and one that doesn't, and predict the
   result before running `keyword_search`.
5. Build `Document`s from any small API or database you have access to, using the pattern from
   Section 1.

---
**Next:** `07_Text_Splitters/` — a Document is often far too big to embed or search usefully. Learn
how to cut it into chunks without cutting a sentence (or an idea) in half.
